# Clark, Abbott & Sompolinsky (2026): the lambda and c grid of the data-derived network

Paper. doi:10.1038/s41593-026-02452-4; claims file `claims/clark-2026.yaml`. Dataset.
DANDI:000939 at version 0.240528.1542, the version the paper cites (see
`tuning_curves.ipynb` for why).

One claim: `figs8-s9-lambda-and-c` (Supplementary Note 3, section 8.4; Supplementary Figs. 8 and
9), run as `data_driven_reconstruction.ipynb` cell 10: lambda in 1e-12 to 1e-2, c in 0 to 1.5,
T 30 s, tau 50 ms, step tau/10 = 5 ms, initial conditions at regularly spaced target angles with
noise of standard deviation 3. The paper's notebook runs 100 initial conditions at each of the
42 grid points; here 20 are run at each point (every 500th target angle), to keep the notebook
within its time limit, and 100 at the points the criterion reads at c 0 and c 1 (a deviation,
recorded in the attempt).

In [ ]:
import time, resource
import numpy as np
import pandas as pd
import hdring as H

T0 = time.time()
sessions, Phi_data, mouse_nums, names = H.load_all_curves()
X, Phi = H.prepare_network_curves(Phi_data)
N = X.shape[1]
print(f"loaded and prepared in {time.time()-T0:.0f} s; curves {Phi_data.shape}; X {X.shape}")

In [ ]:
print("sessions", len(sessions), "| head-direction units", sum(int(d["sess"]["is_hd"].sum()) for d in sessions), "| curves", Phi_data.shape[0])
mu = Phi.mean(0)
U_, s_, Vt_ = np.linalg.svd(Phi - mu, full_matrices=False)
V2 = Vt_[:2].T
ring_pc = (Phi - mu) @ V2
ring_norm = np.linalg.norm(Phi, axis=1).mean()
ring_rad = np.median(np.linalg.norm(ring_pc, axis=1))
print(f"target manifold: mean firing-rate norm {ring_norm:.3f}; top-2 PC radius range {np.linalg.norm(ring_pc, axis=1).min():.3f} to {np.linalg.norm(ring_pc, axis=1).max():.3f}")

## figs8-s9-lambda-and-c

> In the data-derived network, with no uniform inhibition activity diverges or collapses to the
> origin, any c above 0 in the grid prevents the divergence but small c leaves the collapse, a
> large regularization destroys the manifold, and the flow error rises at both large and small
> lambda, with lambda 1e-8 just above its minimum.

Per run: divergence, a non-finite final state or a final firing-rate norm above 10 times the
manifold's (and growing over the last 5 s; a standard choice, as the claims file says only
"non-finite or growing states"). Collapse to the origin, a final firing-rate norm below one
tenth of the manifold's (claims file). The claims file reads collapse in the plane of the top
two principal components of the target manifold, where the origin is the manifold's mean, and
the firing-rate norm of the homogeneous state is not small; so the norm is read two ways:
(i) the norm of the firing rates, as the criterion's words, and (ii) the radius of the projection
on the two principal components, below one tenth of the manifold's median radius there. Flow error, the mean squared flow on the target manifold, which with
the Sinkhorn-normalized targets (stabilizer zero on the manifold) is the reconstruction error
mean((X - Phi J^T)^2) the paper's notebook saves as err_grid.

In [ ]:
lamdas = [1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2]
cs = [0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
def run_point(J, c, n_ic, sd):
    rng = np.random.default_rng(sd)
    step = X.shape[0] // n_ic
    x0 = X[::step][:n_ic].T + rng.standard_normal((N, n_ic)) * 3.0
    norms = {}
    def on_save(t, x):
        norms[round(t, 3)] = np.linalg.norm(H.nonlin(x.astype(np.float64)), axis=0)
    xf = H.simulate(J, x0, T=30.0, dt=0.005, c=c, save_every=5.0, on_save=on_save)
    r = H.nonlin(xf.astype(np.float64))
    fin = np.isfinite(xf).all(0)
    nr = np.where(fin, np.linalg.norm(np.nan_to_num(r), axis=0), np.inf)
    ts = sorted(norms)
    growing = norms[ts[-1]] > 1.01 * norms[ts[-2]]
    diverge = (~fin) | ((nr > 10 * ring_norm) & growing)
    collapse = fin & (nr < 0.1 * ring_norm)
    pc = (np.nan_to_num(r).T - mu) @ V2
    rad = np.linalg.norm(pc, axis=1)
    collapse_pc = fin & (rad < 0.1 * ring_rad)
    return dict(diverge=int(diverge.sum()), collapse=int(collapse.sum()), collapse_pc=int(collapse_pc.sum()), n=n_ic, norm_med=float(np.median(nr[fin])) if fin.any() else np.inf,
                pc_radius_med=float(np.median(rad[fin])) if fin.any() else np.nan)
sd0 = H.seed("figs8-s9-lambda-and-c")
rows, errs = [], {}
t = time.time()
for lam in lamdas:
    J, _ = H.compute_weights(X, Phi, lamda=lam)
    errs[lam] = float(np.mean((X - Phi @ J.T) ** 2))
    for c in cs:
        n_ic = 100 if (c in (0, 1.0) and lam in (1e-12, 1e-8, 1e-4, 1e-2)) else 20
        o = run_point(J, c, n_ic, sd0)
        rows.append(dict(lamda=lam, c=c, **o))
    print(f"lambda {lam:g} done at {time.time()-t:.0f} s", flush=True)
grid = pd.DataFrame(rows)
print(grid.to_string(index=False))
print("flow error (mean squared flow on the manifold) by lambda:", {f"{k:g}": f"{v:.3e}" for k, v in errs.items()})
print(f"manifold firing-rate norm {ring_norm:.3f}; collapse below {0.1*ring_norm:.3f}")
print(f"manifold median radius in the PC plane {ring_rad:.3f}; collapse (ii) below {0.1*ring_rad:.3f}")
c0 = grid[grid.c == 0]
part2 = bool((grid[(grid.c > 0) & (grid.lamda <= 1e-4)].diverge == 0).all())
for col, lab in (("collapse", "(i) firing-rate norm"), ("collapse_pc", "(ii) PC-plane radius")):
    p1 = bool(((c0.diverge + c0[col]) > 0).all()); p1a = bool(((c0.diverge + c0[col]) > 0).any())
    p3 = bool((grid[(grid.c == 1.0) & (grid.lamda == 1e-2)][col] > 0).all())
    p5 = grid[(grid.c == 0.25) & (grid.lamda == 1e-8)][[col, "n"]].values.tolist()
    print(f"reading {lab}: c 0 gives divergence or collapse at every lambda {p1}, at some lambda {p1a}; lambda 1e-2 collapses at c 1: {p3}; collapse at c 0.25, lambda 1e-8: {p5}")
part4 = errs[1e-8] < errs[1e-2] and errs[1e-8] < errs[1e-12]
print("no c above 0 diverges at any lambda up to 1e-4:", part2)
print("flow error at 1e-8 below that at 1e-2 and at 1e-12:", part4, "| lambda of the minimum flow error:", min(errs, key=errs.get))
# seed check: the points the criterion reads, under four more seeds
for k in range(1, 5):
    sd = sd0 + 1000 * k
    line = []
    for lam, c in ((1e-8, 0), (1e-8, 1.0), (1e-2, 1.0), (1e-4, 0.25)):
        J, _ = H.compute_weights(X, Phi, lamda=lam)
        o = run_point(J, c, 20, sd)
        line.append(f"lambda {lam:g} c {c}: diverge {o['diverge']}, collapse (i) {o['collapse']}, (ii) {o['collapse_pc']} of {o['n']}")
    print(f"seed {sd}: " + "; ".join(line))
print(f"time {time.time()-t:.0f} s")

**Result: partially reproduced.** At c 0 some runs diverge at every lambda up to 1e-4 (1 to 4 of
100), and under reading (ii) of collapse nearly all others fall to the homogeneous state (96 to
100 of 100, radius 0.65 to 0.80 against 39 for the manifold) at every lambda; under reading (i) no
run collapses, since the homogeneous state keeps a firing-rate norm of 17.3 to 17.7. No c above 0
diverges up to lambda 1e-4, and c 0.25 still leaves the collapse (12 of 20 at lambda 1e-8 under
reading ii). At lambda 1e-2 the manifold is lost for c up to 0.75 (all 20 collapse under reading
ii), but at c 1 the states keep a radius of 4.6, just above the collapse threshold of 3.9, so the
criterion's test at c 1 fails. The flow error rises at large lambda (2.1e-2 at 1e-2) and at small
lambda from 1.3e-10 at 1e-10 to 2.6e-9 at 1e-12, but 1e-8 (1.7e-8) is not below 1e-12, so that
part of the criterion fails too.

## Summary

| claim | outcome |
|---|---|
| figs8-s9-lambda-and-c | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method. 20 initial conditions at most grid points, 100 at the
points the criterion reads; collapse is read in two ways, as the claims file's words leave open.

In [ ]:
print(f"total time {time.time()-T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")